<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">گزینهٔ عبورکننده از آستانه را گم نکنید</h1>
<p style="text-align:right">درس 63 از 92 · چطور نامزدهای نامحتمل را کنار بگذاریم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">56-topkp</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/56-topkp.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Mask Top-p</bdi> را بنویسید و رفتار <bdi dir="ltr">Top-k</bdi> در امتیازهای مساوی را اصلاح کنید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: مرتب‌سازی،</span> مجموع تجمعی و بازگرداندن اندیس‌ها.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۴۵–۸۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">برای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">[0.6,0.25,0.1,0.05]</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">p=0.8</code> چند گزینه لازم است؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.sampling import filter_logits
torch.set_num_threads(1)
probabilities = torch.tensor([0.6,0.25,0.1,0.05])
print('project candidates:',torch.isfinite(filter_logits(probabilities.log()[None],top_p=0.8)).tolist())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">nucleus_mask(probabilities, p)</code> برای یک بردار احتمال مثبت با مجموع یک و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">0&lt;p&lt;=1</code>، <bdi dir="ltr">Mask</bdi> <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">bool</code> هم‌اندازه در ترتیب اصلی بدهد. کوچک‌ترین پیشوند مرتب با مجموع حداقل <bdi dir="ltr">p</bdi> حفظ شود؛ در تساوی احتمال، ترتیب اصلی حفظ شود.</p>
</div>

In [ ]:
def nucleus_mask(probabilities, p):
    # TODO: مرتب‌سازی، عبور از آستانه، بازگشت به ترتیب اصلی
    return None

In [ ]:
def test_exercise():
    result = nucleus_mask(probabilities,0.8)
    if result is None:
        return False
    assert result.dtype==torch.bool and result.tolist()==[True,True,False,False]
    assert nucleus_mask(probabilities,0.01).tolist()==[True,False,False,False]
    assert nucleus_mask(torch.tensor([0.5,0.5]),0.5).tolist()==[True,False]
    assert nucleus_mask(torch.tensor([0.1,0.6,0.05,0.25]),0.8).tolist()==[False,True,False,True]
    assert nucleus_mask(probabilities,1.0).all()
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: nucleus_mask')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <bdi dir="ltr">p</bdi> را تغییر دهید و تعداد نامزدها را با پیاده‌سازی واقعی اندازه بگیرید؛ این تعداد ثابت نیست.</p>
</div>

In [ ]:
for p in (0.1,0.5,0.8,0.95,1.0):
    mask = torch.isfinite(filter_logits(probabilities.log()[None],top_p=p))
    print(p,mask.tolist(),int(mask.sum()))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">حذف امتیازهای کمتر از امتیازِ گزینهٔ رتبهٔ <bdi dir="ltr">k</bdi>، هنگام تساوی بیش از <bdi dir="ltr">k</bdi> گزینه نگه می‌دارد. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">exact_topk_mask(logits,k)</code> برای بردار یک‌بعدی و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">1&lt;=k&lt;=V</code> <bdi dir="ltr">Mask</bdi>ی با دقیقاً <bdi dir="ltr">k</bdi> خانهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">True</code> بدهد؛ هویت گزینه‌های مساوی در این تمرین مهم نیست.</p>
</div>

In [ ]:
ties = torch.tensor([2.0,2.0,2.0,0.0])
wrong = ties>=ties.topk(2).values[-1]
print('wanted 2, kept:',int(wrong.sum()),wrong.tolist())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def exact_topk_mask(logits, k):
    # TODO: از اندیس گزینه‌ها استفاده کنید
    return None

In [ ]:
def test_repair():
    result = exact_topk_mask(torch.tensor([2.0,2.0,2.0,0.0]),2)
    if result is None:
        return False
    assert result.dtype==torch.bool and result.sum().item()==2
    assert not result[-1]
    assert exact_topk_mask(torch.tensor([0.0,3.0,1.0]),1).tolist()==[False,True,False]
    assert exact_topk_mask(torch.zeros(4),4).all()
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: exact_topk_mask')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">filter_logits</code> در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">sampling.py</code> با همین دو خطای مرزی روبه‌روست. در ترکیب دو روش، ابتدا <bdi dir="ltr">Top-k</bdi> اعمال و سپس <bdi dir="ltr">Top-p</bdi> روی توزیع باقی‌مانده حساب می‌شود.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا <bdi dir="ltr">Mask</bdi> درست باید هم احتمال تجمعی را رعایت کند و هم ترتیب اصلی <bdi dir="ltr">Vocabulary</bdi> را نگه دارد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/56-topkp.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/56-topkp.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>